In [5]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install tensorflow keras pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
pip install OpenCV-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import os
import cv2
import numpy as np
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import VGG16
import seaborn as sns
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Corrected import
from tensorflow.keras.models import Sequential  # Corrected import
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout  # Corrected import
from tensorflow.keras.callbacks import EarlyStopping 

In [10]:
import os
import pandas as pd
from PIL import Image

# List of directories containing images
image_directories = [
    'D:/Knee/train/0Normal',
    'D:/Knee/train/1Doubtful',
    'D:/Knee/train/2Mild',
    'D:/Knee/train/3Moderate',
    'D:/Knee/train/4Severe',
    'D:/Knee/test/0Normal',
    'D:/Knee/test/1Doubtful',
    'D:/Knee/test/2Mild',
    'D:/Knee/test/3Moderate',
    'D:/Knee/test/4Severe'
    # Add more directories as needed
]

# List to hold image data
image_data = []

# Loop through each directory
for image_directory in image_directories:
    # Check if the directory exists
    if os.path.exists(image_directory):
        # Loop through all files in the directory
        for filename in os.listdir(image_directory):
            # Construct full file path
            file_path = os.path.join(image_directory, filename)
            
            # Print the filename being processed
            print(f"Processing file: {file_path}")
            
            # Check if the file is an image
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
                try:
                    # Load the image to ensure it's a valid image file
                    with Image.open(file_path) as img:
                        # Optionally, you can extract image size or other properties
                        width, height = img.size
                        # Append the file information to the list
                        image_data.append({
                            'Filename': filename,
                            'File Path': file_path,
                            'Width': width,
                            'Height': height
                        })
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
            else:
                print(f"Skipped non-image file: {file_path}")
    else:
        print(f"Directory does not exist: {image_directory}")

# Create a DataFrame from the image data
df_images = pd.DataFrame(image_data)

# Display the DataFrame
print("DataFrame contents:")
print(df_images)

Processing file: D:/Knee/train/0Normal\NormalG0 (1).png
Processing file: D:/Knee/train/0Normal\NormalG0 (10).png
Processing file: D:/Knee/train/0Normal\NormalG0 (100).png
Processing file: D:/Knee/train/0Normal\NormalG0 (101).png
Processing file: D:/Knee/train/0Normal\NormalG0 (102).png
Processing file: D:/Knee/train/0Normal\NormalG0 (103).png
Processing file: D:/Knee/train/0Normal\NormalG0 (104).png
Processing file: D:/Knee/train/0Normal\NormalG0 (105).png
Processing file: D:/Knee/train/0Normal\NormalG0 (106).png
Processing file: D:/Knee/train/0Normal\NormalG0 (107).png
Processing file: D:/Knee/train/0Normal\NormalG0 (108).png
Processing file: D:/Knee/train/0Normal\NormalG0 (109).png
Processing file: D:/Knee/train/0Normal\NormalG0 (11).png
Processing file: D:/Knee/train/0Normal\NormalG0 (110).png
Processing file: D:/Knee/train/0Normal\NormalG0 (111).png
Processing file: D:/Knee/train/0Normal\NormalG0 (112).png
Processing file: D:/Knee/train/0Normal\NormalG0 (113).png
Processing file: D

In [11]:
print("DataFrame contents:")
print(df_images.head())

DataFrame contents:
             Filename                                 File Path  Width  Height
0    NormalG0 (1).png    D:/Knee/train/0Normal\NormalG0 (1).png    300     162
1   NormalG0 (10).png   D:/Knee/train/0Normal\NormalG0 (10).png    300     162
2  NormalG0 (100).png  D:/Knee/train/0Normal\NormalG0 (100).png    300     162
3  NormalG0 (101).png  D:/Knee/train/0Normal\NormalG0 (101).png    300     162
4  NormalG0 (102).png  D:/Knee/train/0Normal\NormalG0 (102).png    300     162


In [12]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import train_test_split

# Function to load and preprocess images from multiple directories
def preprocess_images(data_dirs):
    images = []
    labels = []
    categories = ['0Normal', '1Doubtful', '2Mild', '3Moderate','4Severe']  # Adjust based on your dataset structure

    for data_dir in data_dirs:
        for category in categories:
            path = os.path.join(data_dir, category)
            if not os.path.exists(path):
                print(f"Path does not exist: {path}")
                continue
            for img in os.listdir(path):
                img_path = os.path.join(path, img)
                img_array = image.load_img(img_path, target_size=(150, 150))  # Resize images
                img_array = image.img_to_array(img_array)  # Convert to array
                images.append(img_array)
                labels.append(categories.index(category))  # Assign label based on category

    return np.array(images), np.array(labels)


In [13]:

data_dirs = [
    'D:/Knee/train',
    'D:/Knee/test'
]
# Call the preprocess_images function
X, y = preprocess_images(data_dirs)

# Split the dataset into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the images
X_train = X_train / 255.0
X_val = X_val / 255.0


In [15]:
# Define the CNN model
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dense(5, activation='softmax'))  # Change to 4 for multi-class classification

# Compile the model
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',  # Use sparse categorical crossentropy for integer labels
              metrics=['accuracy'])


In [16]:
# Train the model
model.fit(X_train, y_train, epochs=20, validation_data=(X_val, y_val))

Epoch 1/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 19s 217ms/step - accuracy: 0.3155 - loss: 1.5963 - val_accuracy: 0.3894 - val_loss: 1.4651
Epoch 2/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 17s 209ms/step - accuracy: 0.3534 - loss: 1.4519 - val_accuracy: 0.3773 - val_loss: 1.4061
Epoch 3/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 20s 240ms/step - accuracy: 0.3966 - loss: 1.3679 - val_accuracy: 0.4212 - val_loss: 1.3705
Epoch 4/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 20s 241ms/step - accuracy: 0.4845 - loss: 1.2315 - val_accuracy: 0.5273 - val_loss: 1.1366
Epoch 5/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 20s 246ms/step - accuracy: 0.5803 - loss: 1.0362 - val_accuracy: 0.5864 - val_loss: 1.0347
Epoch 6/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 22s 271ms/step - accuracy: 0.6538 - loss: 0.8763 - val_accuracy: 0.6167 - val_loss: 0.8922
Epoch 7/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 23s 276ms/step - accuracy: 0.7470 - loss: 0.6795 - val_accuracy: 0.6773 - val_loss: 0.8242
Epoch 8/20
83/83 ━━━━━━━━━━━━━━━━━━━━ 23s 271ms/step - accuracy: 0.8004 - loss: 0.5315 - val_accu

In [17]:
# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f'Validation Accuracy: {val_accuracy}, Validation Loss: {val_loss}')

21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.8924 - loss: 0.5902
Validation Accuracy: 0.8924242258071899, Validation Loss: 0.5902329683303833


In [18]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

# x_val: validation features
# y_val: true labels (one-hot or integer labels)
# model: trained Keras model

# 1️ Predictions
y_pred_probs = model.predict(X_val)
y_pred = np.argmax(y_pred_probs, axis=1)

# 2️ True labels
if len(y_val.shape) > 1:  # one-hot
    y_true = np.argmax(y_val, axis=1)
else:
    y_true = y_val

# 3️ Basic metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Validation Accuracy: {acc:.4f}")
print(f"Precision (macro): {prec:.4f}")
print(f"Recall (macro): {rec:.4f}")
print(f"F1-score (macro): {f1:.4f}")

# 4️ Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

# 5️ Classification Report
print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# 6️ ROC-AUC (multi-class)
# Convert y_true to one-hot if needed
if len(y_val.shape) == 1:
    from sklearn.preprocessing import label_binarize
    n_classes = len(np.unique(y_true))
    y_true_onehot = label_binarize(y_true, classes=np.arange(n_classes))
else:
    y_true_onehot = y_val

roc_auc = roc_auc_score(y_true_onehot, y_pred_probs, average='macro', multi_class='ovr')
print(f"ROC-AUC (macro, OVR): {roc_auc:.4f}")


21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step
Validation Accuracy: 0.8924
Precision (macro): 0.8913
Recall (macro): 0.8826
F1-score (macro): 0.8862
Confusion Matrix:
[[212   9   2   3   0]
 [ 11 160   9   2   2]
 [  8   2  76   1   0]
 [  2   4   4  70   4]
 [  2   4   2   0  71]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92       226
           1       0.89      0.87      0.88       184
           2       0.82      0.87      0.84        87
           3       0.92      0.83      0.88        84
           4       0.92      0.90      0.91        79

    accuracy                           0.89       660
   macro avg       0.89      0.88      0.89       660
weighted avg       0.89      0.89      0.89       660

ROC-AUC (macro, OVR): 0.9722


In [19]:
# Step 5: Save the model 
model.save('D:\Knee\kneemodel.keras')  # Save the model to a file